<a href="https://colab.research.google.com/github/carolinampessoa/TechChallengeFase5/blob/main/TechChallengeFase5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1) Usar modelo pré-treinado (YOLO / Detectron / etc.)
2) Criar pequeno dataset customizado
3) Fazer fine-tuning
4) Implementar motor de regras STRIDE

In [1]:
!pip install openai

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print("Imagem carregada:", image_path)


In [8]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Digite sua OpenAI API Key: ")

Digite sua OpenAI API Key: ··········


In [9]:
from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print("Imagem carregada:", image_path)


Saving Screenshot_1.png to Screenshot_1 (1).png
Imagem carregada: Screenshot_1 (1).png


In [10]:
from openai import OpenAI
import base64
import json

client = OpenAI()

def encode_image(path):
    with open(path, "rb") as img:
        return base64.b64encode(img.read()).decode("utf-8")

base64_image = encode_image(image_path)

prompt = """
Analise o diagrama de arquitetura de software presente na imagem.

Identifique todos os componentes do sistema.
Classifique cada componente em uma das categorias:

- user
- server
- database
- api
- external_system

Responda APENAS em JSON no formato:

{
  "components": [
    {"name": "...", "type": "..."}
  ]
}
"""

response = client.responses.create(
    model="gpt-4.1",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{base64_image}"
                }
            ]
        }
    ]
)

output_text = response.output_text
print(output_text)


```json
{
  "components": [
    {"name": "Usuários SEI", "type": "user"},
    {"name": "AWS Shield", "type": "external_system"},
    {"name": "Amazon CloudFront", "type": "external_system"},
    {"name": "AWS WAF", "type": "external_system"},
    {"name": "Virtual Private Cloud", "type": "server"},
    {"name": "Application Load Balancer", "type": "server"},
    {"name": "SEI / SIP (API Server)", "type": "server"},
    {"name": "Solr", "type": "server"},
    {"name": "Amazon Elastic File System (NFS) Multi-AZ", "type": "database"},
    {"name": "Amazon RDS (Primary)", "type": "database"},
    {"name": "Amazon RDS (Secondary)", "type": "database"},
    {"name": "Amazon ElastiCache (memcached) Multi-AZ", "type": "database"},
    {"name": "AWS CloudTrail", "type": "external_system"},
    {"name": "AWS Key Management Service", "type": "external_system"},
    {"name": "AWS Backup", "type": "external_system"},
    {"name": "Amazon CloudWatch", "type": "external_system"},
    {"name": "Amazon

In [13]:
import re

# Remove markdown code block delimiters if they exist
json_string = re.search(r'```json\n([\s\S]*?)\n```', output_text)
if json_string:
    clean_output_text = json_string.group(1)
else:
    clean_output_text = output_text.strip()

data = json.loads(clean_output_text)
components = data["components"]

components

[{'name': 'Usuários SEI', 'type': 'user'},
 {'name': 'AWS Shield', 'type': 'external_system'},
 {'name': 'Amazon CloudFront', 'type': 'external_system'},
 {'name': 'AWS WAF', 'type': 'external_system'},
 {'name': 'Virtual Private Cloud', 'type': 'server'},
 {'name': 'Application Load Balancer', 'type': 'server'},
 {'name': 'SEI / SIP (API Server)', 'type': 'server'},
 {'name': 'Solr', 'type': 'server'},
 {'name': 'Amazon Elastic File System (NFS) Multi-AZ', 'type': 'database'},
 {'name': 'Amazon RDS (Primary)', 'type': 'database'},
 {'name': 'Amazon RDS (Secondary)', 'type': 'database'},
 {'name': 'Amazon ElastiCache (memcached) Multi-AZ', 'type': 'database'},
 {'name': 'AWS CloudTrail', 'type': 'external_system'},
 {'name': 'AWS Key Management Service', 'type': 'external_system'},
 {'name': 'AWS Backup', 'type': 'external_system'},
 {'name': 'Amazon CloudWatch', 'type': 'external_system'},
 {'name': 'Amazon Simple Email Service (SES)', 'type': 'external_system'}]

In [14]:
stride_map = {
    "server": ["Spoofing", "Tampering", "Denial of Service"],
    "database": ["Tampering", "Information Disclosure"],
    "api": ["Spoofing", "Repudiation"],
    "user": ["Spoofing"],
    "external_system": ["Spoofing", "Tampering"]
}

def analyze_stride(components):
    results = []

    for comp in components:
        threats = stride_map.get(comp["type"], [])
        results.append({
            "component": comp["name"],
            "type": comp["type"],
            "threats": threats
        })

    return results

stride_results = analyze_stride(components)
stride_results


[{'component': 'Usuários SEI', 'type': 'user', 'threats': ['Spoofing']},
 {'component': 'AWS Shield',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Amazon CloudFront',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'AWS WAF',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Virtual Private Cloud',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Application Load Balancer',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'SEI / SIP (API Server)',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Solr',
  'type': 'server',
  'threats': ['Spoofing', 'Tampering', 'Denial of Service']},
 {'component': 'Amazon Elastic File System (NFS) Multi-AZ',
  'type': 'database',
  'threats': ['Tampering', 'Information Disclosure']},
 {'component'

In [ ]:
def generate_report(results):
    lines = []

    for item in results:
        lines.append(f"Componente: {item['component']}")
        lines.append(f"Tipo: {item['type']}")
        lines.append(f"Ameaças STRIDE: {', '.join(item['threats'])}")
        lines.append("-" * 50)

    return "\n".join(lines)

report = generate_report(stride_results)

print(report)


In [ ]:
counter_prompt = f"""
Considere as seguintes ameaças STRIDE identificadas:

{json.dumps(stride_results, indent=2)}

Sugira contramedidas de segurança para cada componente.
"""

response2 = client.responses.create(
    model="gpt-4.1",
    input=counter_prompt
)

print(response2.output_text)
